In [1]:
# %% [The Definitive Victory: Multi-Moment Spectral Envelopes]
import os
import re
import time
import copy
import warnings
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, TensorDataset
from scipy.fft import fftn, fftshift
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt

# Import classes from your local networks.py
from util import seed_everything, count_parameters
from networks import SpectralViT, SpatialViT, AttentionUNet, SwinTransformer

# --- Config ---
warnings.filterwarnings("ignore", category=UserWarning)
MNI_DIR = os.path.expanduser('~/SpectralViT/data/IXI_extracted/')
CSV_PATH = os.path.expanduser('~/SpectralViT/data/IXI_extracted/IXI.csv')
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.expanduser(f'~/SpectralViT/results/moment_spectral_{RUN_ID}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOL_SIZE = 96
EPOCHS = 200       
N_FOLDS = 5
FRACTIONS = [0.1, 0.25, 0.5, 1.0] 
UNIFIED_LR = 1e-4  

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
seed_everything(0)

# %% [Data Loading]
def load_ixi_data_age(mni_dir, csv_path, vol_size=96):
    df = pd.read_csv(csv_path)
    id_col = [c for c in df.columns if 'ID' in c.upper()][0]
    age_lookup = dict(zip(df[id_col].astype(int), df[[c for c in df.columns if 'AGE' in c.upper() and 'IMAGE' not in c.upper()][0]]))
    files = sorted([f for f in os.listdir(mni_dir) if f.endswith('.nii.gz')])
    vols, ages = [], []
    for f in tqdm(files, desc="Loading Data"):
        match = re.search(r'(\d+)', f)
        if not (match and int(match.group(1)) in age_lookup): continue
        age = age_lookup[int(match.group(1))]
        if pd.isna(age): continue
        img = nib.load(os.path.join(mni_dir, f)).get_fdata()
        c = np.array(img.shape)//2; r = vol_size//2
        crop = img[c[0]-r:c[0]+r, c[1]-r:c[1]+r, c[2]-r:c[2]+r]
        if crop.shape == (vol_size, vol_size, vol_size):
            vols.append((crop - np.mean(crop)) / (np.std(crop) + 1e-8))
            ages.append(float(age))
    return np.array(vols).astype(np.float32), np.array(ages).astype(np.float32)

X, Y = load_ixi_data_age(MNI_DIR, CSV_PATH, VOL_SIZE)

# %% [The Multi-Moment Basis: Mean + Std Radial Profile]
def extract_moment_spectrum(vols, n_shells=64):
    """
    Exploits Natural Ordering while preserving Spectral Texture.
    Basis 1: Mean Magnitude (Energy)
    Basis 2: Std Dev Magnitude (Anisotropy/Texture)
    Total features = 64 * 2 = 128.
    """
    B, D, H, W = vols.shape
    print(f"Extracting {n_shells} Dual-Moment Spectral Envelopes...")
    
    z, y, x = np.indices((D, H, W))
    center = D // 2
    r = np.sqrt((x-center)**2 + (y-center)**2 + (z-center)**2)
    r_flat = r.flatten()
    
    r_max = r.max()
    bins = np.linspace(0, r_max, n_shells + 1)
    bin_indices = np.digitize(r_flat, bins) - 1
    
    all_radial = []
    for i in tqdm(range(B), desc="Moment Mapping"):
        # FFT Magnitude
        mag = np.abs(fftshift(fftn(vols[i], norm='ortho'))).flatten()
        
        means = []
        stds = []
        for b in range(n_shells):
            mask = (bin_indices == b)
            if np.any(mask):
                shell_data = mag[mask]
                means.append(shell_data.mean())
                stds.append(shell_data.std())
            else:
                means.append(0); stds.append(0)
        
        # Concatenate means and stds to maintain ordering: [m0, m1... m63, s0, s1... s63]
        all_radial.append(np.concatenate([means, stds]))
        
    return np.log1p(np.array(all_radial, dtype=np.float32))

X_spectral_raw = extract_moment_spectrum(X)

# %% [Unified Training Function]
def train_model_full(model, train_loader, val_data, y_stats, epochs, lr, device, model_name):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = OneCycleLR(optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)
    criterion = nn.SmoothL1Loss()
    y_mean, y_std = y_stats
    best_mae, best_state, patience, trigger = 100.0, None, 40, 0

    for epoch in range(1, epochs + 1):
        model.train()
        for b_x, b_y in train_loader:
            b_x, b_y = b_x.to(device), b_y.to(device)
            optimizer.zero_grad()
            output = model(b_x); loss = criterion(output, b_y)
            if not torch.isfinite(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                v_x, v_y = val_data
                p = model(v_x.to(device)) if v_x.ndim < 5 else torch.cat([model(c.to(device)) for c in torch.split(v_x, 2)])
                p_np = np.nan_to_num(p.cpu().numpy(), nan=y_mean.item()) * y_std.item() + y_mean.item()
                cur_mae = mean_absolute_error(v_y, p_np)
                if cur_mae < best_mae:
                    best_mae, best_state, trigger = cur_mae, copy.deepcopy(model.state_dict()), 0
                else: trigger += 1
                if trigger >= (patience // 10): break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p = model(v_x.to(device)) if v_x.ndim < 5 else torch.cat([model(c.to(device)) for c in torch.split(v_x, 2)])
        p_np = p.cpu().numpy() * y_std.item() + y_mean.item()
        return {
            'MAE': mean_absolute_error(v_y, p_np), 
            'RMSE': np.sqrt(mean_squared_error(v_y, p_np)), 
            'R2': r2_score(v_y, p_np), 
            'Pearson': pearsonr(v_y.flatten(), p_np.flatten())[0]
        }

# %% [Main Experiment Loop]
results = []
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)

for frac in FRACTIONS:
    print(f"\nREGIME: {int(frac*100)}% DATA")
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        gc.collect(); torch.cuda.empty_cache()
        fold_rng = np.random.RandomState(42 + fold)
        sub_idx = fold_rng.choice(train_idx, int(len(train_idx) * frac), replace=False)
        ym, ys = torch.tensor(Y[sub_idx].mean()), torch.tensor(Y[sub_idx].std() + 1e-8)
        y_norm = ((torch.from_numpy(Y[sub_idx]) - ym) / ys).float()

        # Fold-specific Scaling
        sm, ss = X_spectral_raw[sub_idx].mean(axis=0), X_spectral_raw[sub_idx].std(axis=0) + 1e-8
        tr_s = torch.from_numpy((X_spectral_raw[sub_idx] - sm) / ss).float()
        ts_s = torch.from_numpy((X_spectral_raw[test_idx] - sm) / ss).float()
        tr_v, ts_v = torch.from_numpy(X[sub_idx]).unsqueeze(1).float(), torch.from_numpy(X[test_idx]).unsqueeze(1).float()

        models = {
            "SpectralViT (Moments)": {
                # 128 inputs, 16 tokens (patch 8), embed 32, 2 layers. 
                "model": SpectralViT(n_inputs=128, patch_size=8, embed_dim=32, n_layers=2, n_heads=4).to(device),
                "x": (tr_s, ts_s), "bs": 1
            },
            "SpatialViT (Micro)": {
                "model": SpatialViT(vol_size=32, patch_size=8, embed_dim=24, n_layers=1, n_heads=2).to(device),
                "x": (torch.nn.functional.interpolate(tr_v, size=32), torch.nn.functional.interpolate(ts_v, size=32)), "bs": 1
            },
            "AttnUNet (Micro)": {
                "model": AttentionUNet(in_channels=1, base_channels=8).to(device),
                "x": (tr_v, ts_v), "bs": 1
            },
            "Swin (Micro)": {
                "model": SwinTransformer(img_size=96).to(device),
                "x": (tr_v, ts_v), "bs": 1
            }
        }

        for name, cfg in models.items():
            p_count = count_parameters(cfg['model'])
            loader = DataLoader(TensorDataset(cfg['x'][0], y_norm), batch_size=cfg['bs'], shuffle=True)
            res = train_model_full(cfg['model'], loader, (cfg['x'][1], Y[test_idx]), (ym, ys), EPOCHS, UNIFIED_LR, device, name)
            
            print(f"Fold {fold} | {name:<22} | Params: {p_count:,} | MAE: {res['MAE']:.2f}")
            res.update({'Fraction': frac, 'Fold': fold, 'Model': name, 'Params': p_count})
            results.append(res)
            pd.DataFrame(results).to_csv(os.path.join(OUTPUT_DIR, 'battle_results.csv'), index=False)
            del cfg['model']

# %% [Visualization]
df = pd.DataFrame(results)
fig, axes = plt.subplots(1, 1, figsize=(10, 6))
for model_name in df['Model'].unique():
    sub = df[df['Model'] == model_name].groupby('Fraction')['MAE'].agg(['mean', 'std']).reset_index()
    axes.errorbar(sub['Fraction'], sub['mean'], yerr=sub['std'], label=model_name, marker='o', capsize=5)
axes.set_title("Dual-Moment Spectral ViT vs Spatial Baselines")
axes.set_xlabel("Data Fraction"); axes.set_ylabel("MAE (Years)"); axes.legend(); axes.grid(True, alpha=0.3)
plt.show()

Loading Data: 100%|██████████| 581/581 [01:23<00:00,  6.96it/s]


Extracting 64 Dual-Moment Spectral Envelopes...


Moment Mapping: 100%|██████████| 563/563 [00:33<00:00, 16.96it/s]



REGIME: 10% DATA
Fold 0 | SpectralViT (Moments)  | Params: 21,697 | MAE: 12.00
Fold 0 | SpatialViT (Micro)     | Params: 18,841 | MAE: 12.31
Fold 0 | AttnUNet (Micro)       | Params: 18,724 | MAE: 13.25
Fold 0 | Swin (Micro)           | Params: 19,457 | MAE: 12.10
Fold 1 | SpectralViT (Moments)  | Params: 21,697 | MAE: 10.93
Fold 1 | SpatialViT (Micro)     | Params: 18,841 | MAE: 11.52
Fold 1 | AttnUNet (Micro)       | Params: 18,724 | MAE: 11.94
Fold 1 | Swin (Micro)           | Params: 19,457 | MAE: 10.64
Fold 2 | SpectralViT (Moments)  | Params: 21,697 | MAE: 10.91
Fold 2 | SpatialViT (Micro)     | Params: 18,841 | MAE: 12.84
Fold 2 | AttnUNet (Micro)       | Params: 18,724 | MAE: 14.59
Fold 2 | Swin (Micro)           | Params: 19,457 | MAE: 11.02
Fold 3 | SpectralViT (Moments)  | Params: 21,697 | MAE: 12.79
Fold 3 | SpatialViT (Micro)     | Params: 18,841 | MAE: 15.28
Fold 3 | AttnUNet (Micro)       | Params: 18,724 | MAE: 14.93
Fold 3 | Swin (Micro)           | Params: 19,457 | M

In [ ]:
import numpy as np
import scipy.stats as stats

# 10% regime
spectral_10 = np.array([12.00, 10.93, 10.91, 12.79, 12.65])
spatial_10 = np.array([12.31, 11.52, 12.84, 15.28, 14.03])
attnunet_10 = np.array([13.25, 11.94, 14.59, 14.93, 13.97])
swin_10 = np.array([12.10, 10.64, 11.02, 14.04, 11.72])

# 25% regime
spectral_25 = np.array([12.65, 11.09, 9.87, 12.36, 11.85])
spatial_25 = np.array([12.21, 10.49, 11.28, 12.59, 11.98])
attnunet_25 = np.array([11.88, 13.27, 13.04, 14.14, 14.32])
swin_25 = np.array([11.78, 10.42, 10.51, 11.24, 11.13])

# 50% regime
spectral_50 = np.array([10.66, 9.95, 9.86, 11.15, 11.40])
spatial_50 = np.array([11.65, 10.47, 11.29, 11.83, 12.16])
attnunet_50 = np.array([12.72, 11.54, 13.24, 14.15, 13.33])
swin_50 = np.array([10.52, 10.13, 10.53, 10.91, 11.57])

regimes = {
    '10% Data': {'SpectralViT (Moments)': spectral_10, 'Swin (Micro)': swin_10, 'SpatialViT (Micro)': spatial_10, 'AttnUNet (Micro)': attnunet_10},
    '25% Data': {'SpectralViT (Moments)': spectral_25, 'Swin (Micro)': swin_25, 'SpatialViT (Micro)': spatial_25, 'AttnUNet (Micro)': attnunet_25},
    '50% Data': {'SpectralViT (Moments)': spectral_50, 'Swin (Micro)': swin_50, 'SpatialViT (Micro)': spatial_50, 'AttnUNet (Micro)': attnunet_50}
}

for reg_name, models in regimes.items():
    spec = models['SpectralViT (Moments)']
    print(f"\n--- {reg_name} ---")
    for m_name, vals in models.items():
        mean_val = np.mean(vals)
        if m_name == 'SpectralViT (Moments)':
            print(f"  {m_name}: {mean_val:.3f}")
            continue
        # Does SpectralViT WIN? Meaning SpectralViT < model (spec - vals < 0)
        # Is it statistically significant? i.e., paired t-test where spectral is lower
        t, p_2 = stats.ttest_rel(spec, vals)
        p_1 = p_2 / 2 if t < 0 else 1 - (p_2 / 2)
        wins = mean_val > mean_val # placeholder
        # Spectral wins if spec < vals, so mean(spec) < mean(vals) and t < 0 and p_1 < 0.05
        is_win = (t < 0 and p_1 < 0.05)
        print(f"  {m_name}: {mean_val:.3f} | t={t:.3f}, p_1sided={p_1:.4f} | Win? {is_win}")